### Model training

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
import time

warnings.filterwarnings('ignore')

# Optional: nicer display for pandas in notebooks
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 200)

# Update this path if your CSV is in a different location within the project
csv_path = Path('data/clean/US_Accidents_March23_NY_Clean.csv')
if not csv_path.exists():
    raise FileNotFoundError(f'CSV not found at {csv_path}. Please place the file there or change the path in this cell.')

# Column groups
target_col = 'Severity_Class'
infra_cols = [
    'Amenity','Bump','Crossing','Give_Way','Junction','No_Exit','Railway',
    'Roundabout','Station','Stop','Traffic_Calming','Traffic_Signal','Turning_Loop'
]

numeric_cols = ['Distance(mi)','Temperature(F)','Visibility(mi)',
                        'Humidity(%)','Pressure(in)','Wind_Speed(mph)','Hour','Duration_min']

category_cols = ['Street_encoded', 'County','Weather_Condition','Wind_Direction','Weather_Short','Season']

feature_cols = numeric_cols + category_cols + infra_cols
# Combined categorical columns (infrastructure flags are categorical / indicator features)
use_cols = feature_cols + [target_col]
dtype_map = {
    **{col: 'Int8' for col in infra_cols},
    'Distance(mi)': 'float32',
    'Temperature(F)': 'float32',
    'Visibility(mi)': 'float32',
    'Humidity(%)': 'float32',
    'Pressure(in)': 'float32',
    'Wind_Speed(mph)': 'float32',
    'Hour': 'int16',
    'Duration_min': 'float32',
}

# Read CSV with memory-friendly dtypes
df = pd.read_csv(csv_path, usecols=use_cols, dtype=dtype_map, low_memory=False)
# df = df.head(50000)  # For quicker testing during development; remove or adjust as needed
# Cast categoricals to category for compact storage and safer encoding
df[category_cols] = df[category_cols].apply(lambda s: s.astype('category'))
df[target_col] = df[target_col].astype('category')

print('Loaded:', csv_path, '-> shape:', df.shape)
print('Approx memory MB:', round(df.memory_usage(deep=True).sum() / 1e6, 2))

df.head()


Loaded: data\clean\US_Accidents_March23_NY_Clean.csv -> shape: (307740, 28)
Approx memory MB: 19.72


,Distance(mi),County,Temperature(F),Visibility(mi),Pressure(in),Humidity(%),Weather_Condition,Wind_Direction,Wind_Speed(mph),Amenity,Bump,Crossing,Give_Way,Junction,No_Exit,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Hour,Duration_min,Weather_Short,Season,Severity_Class,Street_encoded
0,0.0100,Rockland,53.099998,1.0,29.809999,93.0,Overcast,East,11.500000,0,0,0,0,0,0,0,0,0,0,0,0,0,16,44.816666,Cloudy/Clear,Autumn,Major,118
1,1.4925,Rockland,53.099998,1.0,29.830000,93.0,Overcast,ENE,8.100000,0,0,0,0,0,0,0,0,0,0,0,0,0,15,89.566666,Cloudy/Clear,Autumn,Major,141
2,1.1700,Westchester,52.000000,2.0,29.809999,93.0,Light Rain,East,17.299999,0,0,0,0,1,0,0,0,0,0,0,0,0,15,89.733330,Rain/Precip,Autumn,Major,1682
3,0.0100,Putnam,52.000000,3.0,29.860001,89.0,Rain,East,10.400000,0,0,0,0,0,0,0,0,0,0,0,0,0,16,44.816666,Rain/Precip,Autumn,Major,1895
4,0.0100,Putnam,52.000000,3.0,29.860001,89.0,Rain,East,10.400000,0,0,0,0,0,0,0,0,0,0,0,0,0,16,44.783333,Rain/Precip,Autumn,Major,1895


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307740 entries, 0 to 307739
Data columns (total 28 columns):
 #   Column             Non-Null Count   Dtype   
---  ------             --------------   -----   
 0   Distance(mi)       307740 non-null  float32 
 1   County             307740 non-null  category
 2   Temperature(F)     307740 non-null  float32 
 3   Visibility(mi)     307740 non-null  float32 
 4   Pressure(in)       307740 non-null  float32 
 5   Humidity(%)        307740 non-null  float32 
 6   Weather_Condition  307740 non-null  category
 7   Wind_Direction     307740 non-null  category
 8   Wind_Speed(mph)    307740 non-null  float32 
 9   Amenity            307740 non-null  Int8    
 10  Bump               307740 non-null  Int8    
 11  Crossing           307740 non-null  Int8    
 12  Give_Way           307740 non-null  Int8    
 13  Junction           307740 non-null  Int8    
 14  No_Exit            307740 non-null  Int8    
 15  Railway            307740 non-null

In [3]:
# Combined categorical columns (infrastructure flags are categorical / indicator features)
print('Categorical columns', category_cols + infra_cols)
print('Numeric columns', numeric_cols)
print('Target column', target_col)


Categorical columns ['Street_encoded', 'County', 'Weather_Condition', 'Wind_Direction', 'Weather_Short', 'Season', 'Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal', 'Turning_Loop']
Numeric columns ['Distance(mi)', 'Temperature(F)', 'Visibility(mi)', 'Humidity(%)', 'Pressure(in)', 'Wind_Speed(mph)', 'Hour', 'Duration_min']
Target column Severity_Class


In [4]:
df.isna().sum()

Distance(mi)         0
County               0
Temperature(F)       0
Visibility(mi)       0
Pressure(in)         0
Humidity(%)          0
Weather_Condition    0
Wind_Direction       0
Wind_Speed(mph)      0
Amenity              0
Bump                 0
Crossing             0
Give_Way             0
Junction             0
No_Exit              0
Railway              0
Roundabout           0
Station              0
Stop                 0
Traffic_Calming      0
Traffic_Signal       0
Turning_Loop         0
Hour                 0
Duration_min         0
Weather_Short        0
Season               0
Severity_Class       0
Street_encoded       0
dtype: int64

In [5]:
df.head(5)

,Distance(mi),County,Temperature(F),Visibility(mi),Pressure(in),Humidity(%),Weather_Condition,Wind_Direction,Wind_Speed(mph),Amenity,Bump,Crossing,Give_Way,Junction,No_Exit,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Hour,Duration_min,Weather_Short,Season,Severity_Class,Street_encoded
0,0.0100,Rockland,53.099998,1.0,29.809999,93.0,Overcast,East,11.500000,0,0,0,0,0,0,0,0,0,0,0,0,0,16,44.816666,Cloudy/Clear,Autumn,Major,118
1,1.4925,Rockland,53.099998,1.0,29.830000,93.0,Overcast,ENE,8.100000,0,0,0,0,0,0,0,0,0,0,0,0,0,15,89.566666,Cloudy/Clear,Autumn,Major,141
2,1.1700,Westchester,52.000000,2.0,29.809999,93.0,Light Rain,East,17.299999,0,0,0,0,1,0,0,0,0,0,0,0,0,15,89.733330,Rain/Precip,Autumn,Major,1682
3,0.0100,Putnam,52.000000,3.0,29.860001,89.0,Rain,East,10.400000,0,0,0,0,0,0,0,0,0,0,0,0,0,16,44.816666,Rain/Precip,Autumn,Major,1895
4,0.0100,Putnam,52.000000,3.0,29.860001,89.0,Rain,East,10.400000,0,0,0,0,0,0,0,0,0,0,0,0,0,16,44.783333,Rain/Precip,Autumn,Major,1895


In [6]:
# Work on a copy to be safe (kept as a note, but avoided here to save memory)
# Ensure expected dtypes remain compact
for col in infra_cols:
    if str(df[col].dtype) != 'Int8':
        df[col] = df[col].astype('Int8')

for col in category_cols:
    if df[col].dtype.name != 'category':
        df[col] = df[col].astype('category')

df[target_col] = df[target_col].astype('category')
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307740 entries, 0 to 307739
Data columns (total 28 columns):
 #   Column             Non-Null Count   Dtype   
---  ------             --------------   -----   
 0   Distance(mi)       307740 non-null  float32 
 1   County             307740 non-null  category
 2   Temperature(F)     307740 non-null  float32 
 3   Visibility(mi)     307740 non-null  float32 
 4   Pressure(in)       307740 non-null  float32 
 5   Humidity(%)        307740 non-null  float32 
 6   Weather_Condition  307740 non-null  category
 7   Wind_Direction     307740 non-null  category
 8   Wind_Speed(mph)    307740 non-null  float32 
 9   Amenity            307740 non-null  Int8    
 10  Bump               307740 non-null  Int8    
 11  Crossing           307740 non-null  Int8    
 12  Give_Way           307740 non-null  Int8    
 13  Junction           307740 non-null  Int8    
 14  No_Exit            307740 non-null  Int8    
 15  Railway            307740 non-null

In [8]:
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix, classification_report

In [9]:
# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# --------------------------------------------------------------------------
# 1) Train/test split
# - Stratify by target so class proportions are preserved in train/test
# - This prevents accidental optimistic evaluation when classes are imbalanced
# --------------------------------------------------------------------------
X = df[feature_cols]
y = df[target_col]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)


In [10]:
# --------------------------------------------------------------------------
# 2) Preprocessing pipelines
# - Numeric: median imputation + standard scaling (robust and works for most models)
# - Categorical & infra flags: most-frequent imputation + one-hot encoding
# - Keep preprocessing inside a pipeline to avoid train-test leakage and to make CV safe
# - Sparse-friendly: scaling without centering and sparse OHE to reduce memory
# --------------------------------------------------------------------------

num_cols = numeric_cols + infra_cols

num_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler(with_mean=False))  # keep sparse representation
])

cat_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=True))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, category_cols)
], remainder='drop', sparse_threshold=0.0)  # always return sparse matrix


In [11]:
df[category_cols].info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307740 entries, 0 to 307739
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype   
---  ------             --------------   -----   
 0   Street_encoded     307740 non-null  category
 1   County             307740 non-null  category
 2   Weather_Condition  307740 non-null  category
 3   Wind_Direction     307740 non-null  category
 4   Weather_Short      307740 non-null  category
 5   Season             307740 non-null  category
dtypes: category(6)
memory usage: 2.1 MB


In [12]:
# --------------------------------------------------------------------------
# 3) Baseline models
# - LogisticRegression: interpretable baseline with class_weight="balanced"
# - DecisionTree: simple nonlinear baseline
# - RandomForest: ensemble baseline capturing interactions and non-linearities
# Each model is wrapped in a pipeline that includes the preprocessor
# - Sparse-friendly configuration for lower memory
# --------------------------------------------------------------------------

lr_pipeline = Pipeline([
    ('preproc', preprocessor),
    ('clf', LogisticRegression(solver='saga', class_weight='balanced', multi_class='multinomial', max_iter=500, n_jobs=4, random_state=RANDOM_SEED))
])

dt_pipeline = Pipeline([
    ('preproc', preprocessor),
    ('clf', DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_SEED))
])

rf_pipeline = Pipeline([
    ('preproc', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=RANDOM_SEED, n_jobs=4))
])


In [13]:
# Fit baselines and print evaluation (weighted F1 is primary metric)
print("Fitting baseline models...")
for name, pipe in [('LogisticRegression', lr_pipeline), ('DecisionTree', dt_pipeline), ('RandomForest', rf_pipeline)]:
    pipe.fit(X_train, y_train)                       # train model
    preds = pipe.predict(X_test)                     # predict on test set
    print(f"\n=== {name} ===")
    print("Weighted F1:", f1_score(y_test, preds, average='weighted'))  # primary metric
    print(classification_report(y_test, preds))     # precision/recall per class;

Fitting baseline models...

=== LogisticRegression ===
Weighted F1: 0.6937332228564134
              precision    recall  f1-score   support

       Major       0.57      0.58      0.58     15861
       Minor       0.02      0.92      0.03       248
    Moderate       0.89      0.63      0.74     45439

    accuracy                           0.62     61548
   macro avg       0.49      0.71      0.45     61548
weighted avg       0.80      0.62      0.69     61548


=== DecisionTree ===
Weighted F1: 0.8377489675689227
              precision    recall  f1-score   support

       Major       0.68      0.71      0.70     15861
       Minor       0.33      0.48      0.39       248
    Moderate       0.90      0.88      0.89     45439

    accuracy                           0.84     61548
   macro avg       0.64      0.69      0.66     61548
weighted avg       0.84      0.84      0.84     61548


=== RandomForest ===
Weighted F1: 0.8656867038956748
              precision    recall  f1-score

In [14]:
#  pip install xgboost

In [15]:
import xgboost as xgb

In [16]:
# -----------------------
# XGBoost pipeline (preprocessing -> XGB classifier)
# - We use sklearn wrapper XGBClassifier so it fits into Pipeline
# - Use histogram tree builder and modest params to stay memory efficient
# -----------------------
xgb_pipeline = Pipeline([
    ('preproc', preprocessor),
    ('clf', xgb.XGBClassifier(
        objective='multi:softprob',
        eval_metric='mlogloss',
        random_state=RANDOM_SEED,
        use_label_encoder=False,
        tree_method='hist',
        max_depth=6,
        n_estimators=400,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=4
    ))
])


In [17]:
# -----------------------
# Option: train with SMOTE (example)
# - Use imblearn Pipeline so SMOTE is applied only to training folds
# -----------------------
use_smote = False  # set True to try oversampling
if use_smote:
    xgb_smote_pipeline = ImbPipeline([
        ('preproc', preprocessor),
        ('smote', SMOTE(random_state=RANDOM_SEED)),
        ('clf', xgb.XGBClassifier(objective='multi:softprob', random_state=RANDOM_SEED, use_label_encoder=False, eval_metric='mlogloss', n_jobs=-1))
    ])

In [18]:
# -----------------------
# Quick baseline fit for XGBoost (default params)
# -----------------------
# print("\nFitting XGBoost baseline (default params)...")

# xgb_pipeline.fit(X_train, y_train)
# xgb_preds = xgb_pipeline.predict(X_test)
# print("XGBoost baseline weighted F1:", f1_score(y_test, xgb_preds, average='weighted'))
# print(classification_report(y_test, xgb_preds))

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report

print("\nFitting XGBoost baseline (default params)...")

# encode labels (fit only on training labels)
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# fit and predict with encoded labels
xgb_pipeline.fit(X_train, y_train_enc)
xgb_preds_enc = xgb_pipeline.predict(X_test)

# decode predictions back to original string labels for readable reports
xgb_preds = le.inverse_transform(xgb_preds_enc)
y_test_labels = le.inverse_transform(y_test_enc)

print("XGBoost baseline weighted F1:", f1_score(y_test_labels, xgb_preds, average='weighted'))
print(classification_report(y_test_labels, xgb_preds))


Fitting XGBoost baseline (default params)...
XGBoost baseline weighted F1: 0.8710065262537967
              precision    recall  f1-score   support

       Major       0.83      0.67      0.74     15861
       Minor       0.73      0.22      0.34       248
    Moderate       0.89      0.95      0.92     45439

    accuracy                           0.88     61548
   macro avg       0.82      0.61      0.67     61548
weighted avg       0.87      0.88      0.87     61548



In [19]:
#  !pip install catboost

In [20]:
# -----------------------
# Advanced model: CatBoost (handles categorical features efficiently)
# -----------------------
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import f1_score, classification_report

# Identify categorical feature indices for CatBoost (raw, not one-hot)
cat_feature_indices = [df[feature_cols].columns.get_loc(col) for col in category_cols]

# Create CatBoost Pools for training/testing
train_pool = Pool(df.loc[X_train.index, feature_cols], y_train, cat_features=cat_feature_indices)
test_pool = Pool(df.loc[X_test.index, feature_cols], y_test, cat_features=cat_feature_indices)

cat_model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',
    depth=8,
    learning_rate=0.1,
    iterations=300,
    l2_leaf_reg=3,
    bootstrap_type='Bernoulli',
    subsample=0.8,
    random_seed=RANDOM_SEED,
    verbose=100
)

cat_model.fit(train_pool, eval_set=test_pool, use_best_model=True)

cat_preds = cat_model.predict(test_pool).ravel()
print('CatBoost weighted F1:', f1_score(y_test, cat_preds, average='weighted'))
print(classification_report(y_test, cat_preds))

# # Optional: feature importance (LossFunctionChange gives stable global importance)
# cat_importance = cat_model.get_feature_importance(type='LossFunctionChange')
# importance_df = (
#     pd.DataFrame({'feature': feature_cols, 'importance': cat_importance})
#     .sort_values('importance', ascending=False)
# )
# print('Top 10 important features:')
# print(importance_df.head(10))


0:	learn: 0.8202303	test: 0.8225368	best: 0.8225368 (0)	total: 1.05s	remaining: 5m 15s
100:	learn: 0.8758518	test: 0.8781536	best: 0.8781536 (100)	total: 1m 34s	remaining: 3m 6s
200:	learn: 0.8855717	test: 0.8856357	best: 0.8856357 (200)	total: 3m 13s	remaining: 1m 35s
299:	learn: 0.8900065	test: 0.8885954	best: 0.8885954 (299)	total: 4m 59s	remaining: 0us

bestTest = 0.8885954309
bestIteration = 299

CatBoost weighted F1: 0.8885954309377956
              precision    recall  f1-score   support

       Major       0.83      0.74      0.78     15861
       Minor       0.64      0.13      0.21       248
    Moderate       0.91      0.95      0.93     45439

    accuracy                           0.89     61548
   macro avg       0.79      0.61      0.64     61548
weighted avg       0.89      0.89      0.89     61548



In [21]:
# -----------------------
# Hyperparameter tuning (RandomizedSearchCV)
# - StratifiedKFold preserves class proportions across folds
# - Optimize for weighted F1 (primary metric)
# - Keep search compact for speed; expand ranges for full tuning
# - Encodes string labels once to keep XGBoost happy
# -----------------------

import time
from sklearn.preprocessing import LabelEncoder

param_dist = {
    'clf__max_depth': [4, 6, 8],
    'clf__learning_rate': [0.01, 0.03, 0.1],
    'clf__n_estimators': [100, 300, 600],
    'clf__subsample': [0.6, 0.8, 1.0],
    'clf__colsample_bytree': [0.6, 0.8, 1.0]
}

# Encode labels to integers once
le = LabelEncoder()
le.fit(y_train)
y_train_enc = le.transform(y_train)
y_test_enc = le.transform(y_test)

# Sample training data for tuning to save memory
sample_n = 80000
if len(X_train) > sample_n:
    X_tune, _, y_tune, _ = train_test_split(
        X_train, y_train, train_size=sample_n, stratify=y_train, random_state=RANDOM_SEED
    )
    y_tune_enc = le.transform(y_tune)
else:
    X_tune, y_tune = X_train, y_train
    y_tune_enc = y_train_enc

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
rand_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_dist,
    n_iter=5,
    scoring='f1_weighted',
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbose=1,
    error_score='raise'
)

print("Running RandomizedSearchCV for XGBoost...")
t0 = time.time()
rand_search.fit(X_tune, y_tune_enc)
t1 = time.time()
print(f"RandomizedSearchCV done in {t1 - t0:.1f}s")

best_xgb = rand_search.best_estimator_
y_pred_enc = best_xgb.predict(X_test)
y_pred = le.inverse_transform(y_pred_enc)

print('Best params:', rand_search.best_params_)
print('Best CV weighted F1:', rand_search.best_score_)
from sklearn.metrics import f1_score
print('Test weighted F1:', f1_score(y_test_enc, y_pred_enc, average='weighted'))


Running RandomizedSearchCV for XGBoost...
Fitting 5 folds for each of 5 candidates, totalling 25 fits
RandomizedSearchCV done in 842.5s
Best params: {'clf__subsample': 0.8, 'clf__n_estimators': 600, 'clf__max_depth': 6, 'clf__learning_rate': 0.1, 'clf__colsample_bytree': 1.0}
Best CV weighted F1: 0.8673306789018185
Test weighted F1: 0.8707655224044164


In [22]:
# Reuse the memory-friendly xgb_pipeline defined earlier (cell 16)
xgb_pipeline


Pipeline(steps=[('preproc',
                 ColumnTransformer(sparse_threshold=0.0,
                                   transformers=[('num',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scale',
                                                                   StandardScaler(with_mean=False))]),
                                                  ['Distance(mi)',
                                                   'Temperature(F)',
                                                   'Visibility(mi)',
                                                   'Humidity(%)',
                                                   'Pressure(in)',
                                                   'Wind_Speed(mph)', 'Hour',
                                                   'Duration_min', 'Amenity',
                                                   'Bump', 'Crossing',
                                                   'Give_Way', '...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.1,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=6, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=400, n_jobs=4,
                               num_parallel_tree=None, ...))])

In [23]:
# -----------------------
# Model evaluation, fairness slices, and model retention
# Primary metric: weighted F1. Also reports precision/recall per class, ROC-AUC (ovo), confusion matrix.
# Fairness: slices by urban proxy, county, and hour buckets. Uses sampling already defined upstream to avoid memory blowup.
# -----------------------
from pathlib import Path
import numpy as np
import joblib
from sklearn.metrics import (
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.preprocessing import label_binarize

# Build a compact test DataFrame for slicing (avoid copying full df)
df_test = df.loc[X_test.index, feature_cols].copy()
# Hour buckets for time-based fairness
if 'Hour' in df_test.columns:
    hour_vals = df_test['Hour']  # squeeze removes extra dimensions
    # print(hour_vals)
    hour_vals = pd.Series(df_test['Hour'].values, index=df_test.index)
    df_test['hour_bucket'] = pd.cut(hour_vals, bins=[-1, 6, 10, 16, 20, 24],
                                    labels=['overnight', 'morning', 'day', 'evening', 'night'])
# Simple urban/rural proxy using traffic signals
if 'Traffic_Signal' in df_test.columns:
    df_test['urban_proxy'] = np.where(df_test['Traffic_Signal'] > 0, 'urban_like', 'rural_like')




In [24]:
def fairness_report(df_slice, y_true, y_pred, group_col, top_n=5):
    print("fairness_report")
    if group_col not in df_slice.columns:
        print(f"Skipped fairness slice for {group_col} (column missing)")
        return
    counts = df_slice[group_col].value_counts().head(top_n)
    if counts.empty:
        print(f"No data for {group_col}")
        return
    print(f"Fairness slice by {group_col} (top {len(counts)} groups)")
    for grp, n in counts.items():
        mask = df_slice[group_col] == grp
        if mask.sum() == 0:
            continue
        f1 = f1_score(y_true[mask], y_pred[mask], average='weighted')
        print(f"  {group_col}={grp}: n={n}, weighted F1={f1:.3f}")


def evaluate_model(model, X_eval, y_eval, model_name='model', label_encoder=None):
    if model_name=='catboost':
        preds = model.predict(X_eval).ravel()
    else:
        preds = model.predict(X_eval)
        
    if label_encoder is not None:
        y_labels = label_encoder.inverse_transform(y_eval)
        pred_labels = label_encoder.inverse_transform(preds)
    else:
        y_labels = y_eval
        pred_labels = preds

    # Primary metric
    weighted_f1 = f1_score(y_labels, pred_labels, average='weighted')
    print(f"{model_name} weighted F1: {weighted_f1:.3f}")

    # Per-class precision/recall/F1 (focus on severe classes)
    labels = np.unique(y_labels)
    prec, rec, f1, support = precision_recall_fscore_support(
        y_labels, pred_labels, labels=labels, zero_division=0
    )
    severe_classes = [c for c in labels if c in ['Major', 'Minor', 'Moderate']]
    print("Per-class precision/recall/F1 (severe focus):")
    for lbl, p, r, f, n in zip(labels, prec, rec, f1, support):
        if lbl in severe_classes:
            print(f"  {lbl:8s} P={p:.3f} R={r:.3f} F1={f:.3f} n={n}")

    # Confusion matrix
    cm = confusion_matrix(y_labels, pred_labels, labels=labels)
    print("Confusion matrix (labels order):", list(labels))
    print(cm)

    # ROC-AUC (ovo, weighted)
    try:
        y_true_bin = label_binarize(y_labels, classes=labels)
        y_pred_bin = label_binarize(pred_labels, classes=labels)
        roc_auc = roc_auc_score(y_true_bin, y_pred_bin, average='weighted', multi_class='ovo')
        print("ROC-AUC (weighted ovo):", round(roc_auc, 3))
    except Exception as e:
        print("ROC-AUC not computed:", e)

    # Fairness slices
    print("\nFairness reports:",model_name)
    y_true_series = pd.Series(y_labels, index=df_test.index)
    y_pred_series = pd.Series(pred_labels, index=df_test.index)
    fairness_report(df_test, y_true_series, y_pred_series, 'urban_proxy')
    fairness_report(df_test, y_true_series, y_pred_series, 'County')
    fairness_report(df_test, y_true_series, y_pred_series, 'hour_bucket')

    return weighted_f1





In [25]:
# Evaluate available models and pick the best for retention
candidate_scores = []
if 'best_xgb' in globals():
    score = evaluate_model(best_xgb, X_test, y_test_enc, model_name='best_xgb (XGBoost)', label_encoder=le)
    candidate_scores.append(('best_xgb', best_xgb, score))
elif 'xgb_baseline' in globals():
    score = evaluate_model(xgb_baseline, X_test, y_test, model_name='xgb_baseline')
    candidate_scores.append(('xgb_baseline', xgb_baseline, score))

if 'cat_model' in globals():
    print(X_test[feature_cols].shape)   # should be (n_samples, n_features)
    score = evaluate_model(cat_model, df.loc[X_test.index, feature_cols], y_test, model_name='catboost')
    candidate_scores.append(('catboost', cat_model, score))

if candidate_scores:
    final_name, final_model, final_score = sorted(candidate_scores, key=lambda t: t[2], reverse=True)[0]
    models_dir = Path('models')
    cbm_path = models_dir / f'{final_name}.cbm'
    try:
            final_model.save_model(cbm_path)
            print(f"Saved CBM model to {cbm_path}")
    except Exception as e:
            print(f"Could not save CBM model: {e}")
    models_dir.mkdir(parents=True, exist_ok=True)
    out_path = models_dir / f'{final_name}.joblib'

    joblib.dump(final_model, out_path)
    print(f"Saved final model '{final_name}' to {out_path} (weighted F1={final_score:.3f})")
else:
    print("No candidate models available to save.")

best_xgb (XGBoost) weighted F1: 0.871
Per-class precision/recall/F1 (severe focus):
  Major    P=0.816 R=0.680 F1=0.742 n=15861
  Minor    P=0.746 R=0.214 F1=0.332 n=248
  Moderate P=0.892 R=0.947 F1=0.919 n=45439
Confusion matrix (labels order): ['Major', 'Minor', 'Moderate']
[[10788     2  5071]
 [   51    53   144]
 [ 2385    16 43038]]
ROC-AUC (weighted ovo): 0.811

Fairness reports: best_xgb (XGBoost)
fairness_report
Fairness slice by urban_proxy (top 2 groups)
  urban_proxy=rural_like: n=52170, weighted F1=0.867
  urban_proxy=urban_like: n=9378, weighted F1=0.891
fairness_report
Fairness slice by County (top 5 groups)
  County=Queens: n=7125, weighted F1=0.864
  County=Westchester: n=6934, weighted F1=0.872
  County=Monroe: n=6082, weighted F1=0.895
  County=Bronx: n=4766, weighted F1=0.866
  County=Suffolk: n=4081, weighted F1=0.866
fairness_report
Fairness slice by hour_bucket (top 5 groups)
  hour_bucket=day: n=19768, weighted F1=0.879
  hour_bucket=morning: n=16727, weighted 

In [26]:
cat_model.feature_names_

['Distance(mi)',
 'Temperature(F)',
 'Visibility(mi)',
 'Humidity(%)',
 'Pressure(in)',
 'Wind_Speed(mph)',
 'Hour',
 'Duration_min',
 'Street_encoded',
 'County',
 'Weather_Condition',
 'Wind_Direction',
 'Weather_Short',
 'Season',
 'Amenity',
 'Bump',
 'Crossing',
 'Give_Way',
 'Junction',
 'No_Exit',
 'Railway',
 'Roundabout',
 'Station',
 'Stop',
 'Traffic_Calming',
 'Traffic_Signal',
 'Turning_Loop']

In [27]:
expected_features = cat_model.feature_names_
input_df = df[expected_features]
input_df

,Distance(mi),Temperature(F),Visibility(mi),Humidity(%),Pressure(in),Wind_Speed(mph),Hour,Duration_min,Street_encoded,County,Weather_Condition,Wind_Direction,Weather_Short,Season,Amenity,Bump,Crossing,Give_Way,Junction,No_Exit,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop
0,0.0100,53.099998,1.0,93.0,29.809999,11.500000,16,44.816666,118,Rockland,Overcast,East,Cloudy/Clear,Autumn,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1.4925,53.099998,1.0,93.0,29.830000,8.100000,15,89.566666,141,Rockland,Overcast,ENE,Cloudy/Clear,Autumn,0,0,0,0,0,0,0,0,0,0,0,0,0
2,1.1700,52.000000,2.0,93.0,29.809999,17.299999,15,89.733330,1682,Westchester,Light Rain,East,Rain/Precip,Autumn,0,0,0,0,1,0,0,0,0,0,0,0,0
3,0.0100,52.000000,3.0,89.0,29.860001,10.400000,16,44.816666,1895,Putnam,Rain,East,Rain/Precip,Autumn,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0.0100,52.000000,3.0,89.0,29.860001,10.400000,16,44.783333,1895,Putnam,Rain,East,Rain/Precip,Autumn,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307735,0.0000,75.000000,10.0,51.0,29.969999,10.000000,16,27.566668,6635,Queens,Cloudy,N,Cloudy/Clear,Summer,0,0,0,0,0,0,0,0,0,0,0,0,0
307736,0.0000,75.000000,10.0,50.0,29.530001,7.000000,16,27.916666,8,Westchester,Partly Cloudy,NNW,Cloudy/Clear,Summer,0,0,0,1,0,0,0,0,0,0,0,1,0
307737,0.0750,73.000000,10.0,64.0,29.830000,8.000000,16,25.950001,2954,Nassau,Fair,N,Cloudy/Clear,Summer,0,0,0,0,1,0,0,0,0,0,0,0,0
307738,1.4925,75.000000,10.0,57.0,29.969999,6.000000,19,28.950001,1533,Richmond,Mostly Cloudy,VAR,Cloudy/Clear,Summer,0,0,0,0,0,0,0,0,0,0,0,0,0
